# Counting vehicles by class for signal timing (UVH-26, YOLOv8)

AIML ZG525 Computer Vision, EC-1 Assignment 1 · Track B (object detection)

This notebook runs top to bottom on a free Colab T4 (*Runtime → Change runtime type → T4 GPU*).

- **`RUN_TRAINING`** (next cell): `True` retrains both models; `False` skips training and evaluates the weights saved on Drive.
- The dataset is downloaded by the notebook from the Hugging Face Hub at a pinned revision; no data files are embedded.
- Sections follow the report's 8 steps. Every number quoted in the report appears in a cell output below.
- Optional Colab secrets: `HF_TOKEN` (faster Hub downloads), `GITHUB_TOKEN` (only if the repo is private).

In [ ]:
# ---- Run settings: the only cell a grader should need to edit ----
RUN_TRAINING = True   # False: skip training, evaluate the weights saved under DRIVE_ROOT/runs
REPO_URL = "https://github.com/YashRenwa/uvh26-pcu-counting.git"
COMMIT = "main"       # pinned to a commit hash for the final submission run
DRIVE_ROOT = "/content/drive/MyDrive/uvh26"

In [ ]:
import os, subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules


def sh(cmd):
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=True)


if IN_COLAB:
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    repo_dir = Path("/content/uvh26-pcu-counting")
    if not repo_dir.exists():
        try:
            token = userdata.get("GITHUB_TOKEN")  # only needed for a private repo
        except Exception:
            token = None
        url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL
        subprocess.run(f"git clone -q {url} {repo_dir}", shell=True, check=True)
    os.chdir(repo_dir)
    sh(f"git fetch -q origin && git checkout -q {COMMIT} && (git merge -q --ff-only origin/{COMMIT} 2>/dev/null || true)")
    # Keep Colab's preinstalled torch/opencv/numpy (CUDA-matched, no restart); pin the rest.
    sh("grep -v colab-skip requirements.txt > /tmp/req-colab.txt && pip install -q -r /tmp/req-colab.txt")
    os.environ.update({
        "UVH_RAW_ROOT": "/content/raw",              # original PNGs on fast local disk
        "UVH_DATA_ROOT": f"{DRIVE_ROOT}/prepared",   # JPEGs + labels survive a disconnect
        "UVH_RUNS_ROOT": f"{DRIVE_ROOT}/runs",
        "UVH_WEIGHTS_DIR": f"{DRIVE_ROOT}/weights",
        "UVH_DATASETS_ROOT": "/content/datasets",   # training reads from local disk, not Drive
    })
else:
    repo_dir = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    os.chdir(repo_dir)

sys.path.insert(0, str(repo_dir))
commit = subprocess.run("git rev-parse --short HEAD", shell=True, capture_output=True, text=True).stdout.strip()
print("repo:", repo_dir, "| commit:", commit or "(no commits yet)")

In [ ]:
import torch
from src.common import library_versions, load_config, set_seed

SMOKE = os.environ.get("UVH_SMOKE") == "1"  # local CPU smoke test on 20 images; never set on Colab
cfg = load_config(smoke=SMOKE)
set_seed(cfg["seed"])
print("seed:", cfg["seed"])
for name, version in library_versions().items():
    print(f"  {name:16s} {version}")
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
print("paths:", {k: str(v) for k, v in cfg["paths"].items()})

## 1. Problem and task type

*Written with the report.* Signal timing needs per-class vehicle **counts** at a junction, converted to PCU. Counting needs instances, so this is object detection: classification cannot count, and segmentation adds labelling and compute cost without changing the count.

## 2. Data and split  *(Phase 1)*

**Source.** UVH-26 (IISc AIM with the Bengaluru Traffic Police), **CC BY 4.0**, downloaded from the Hugging Face Hub at the revision pinned in `configs/base.yaml`. The full set is 26,646 PNGs (90 GB), so only the four annotation JSONs and a seeded pool of 4,000 images are fetched (`splits/pool.csv`).

**Pool.** STAPLE labels exist for only 81% of images (MV covers all), so the pool is drawn from images that carry both label sets. The MV and STAPLE runs then train on identical images.

**Pseudo-cameras.** UVH-26 has no camera ids, and a fixed CCTV camera produces many near-identical frames. Two images are linked when the correlation **g** of their 64×36 grayscale *gradient-magnitude* thumbnails is ≥ 0.48; connected components are pseudo-cameras. A fixed camera keeps its static edges at the same pixels, so g survives lighting, glare and colour-vs-IR changes. CLIP ViT-B/32 cosine and intensity correlation did not: CLIP alone chained different cameras into giant groups, and on 88 hand-labelled hard pairs g separated cameras best (AUC 0.97 vs 0.80 for CLIP).

**Split.** Whole pseudo-cameras are split 60/20/20 with the best class balance among 50 seeds, then a **purge** moves any val/test image with a train image at g ≥ 0.36 (and its pseudo-camera) into train, repeated until none is left. A random audit of 40 test images still finds a same-camera train image for about 7 of them, reported as residual leakage.

**Frozen split.** The split was built once and committed (`splits/`, checksum in `splits/test.sha256`). CLIP outputs differ slightly between CPU and GPU, so this notebook recomputes the grouping to show the evidence, but trains and evaluates on the committed split after verifying the test-set checksum.

In [ ]:
import pandas as pd
from IPython.display import Image, display
from src import cluster, download, prepare, split

FIGS = cfg["paths"]["results_dir"] / "figures"
pool = download.select_pool(cfg)   # reads splits/pool.csv from the repo
print(f"pool: {len(pool)} images with both MV and STAPLE labels")
print(pool.groupby(["official_split", "subfolder"]).size().rename("images").to_string())
download.download_pool(cfg, pool)  # PNGs to local disk; skips images already prepared on Drive

Resize to 1280 px on the long side (JPEG quality 95) and write YOLO labels twice, once from MV and once from STAPLE. YOLO box coordinates are normalised, so they need no rescaling.

In [ ]:
prep = prepare.run(cfg, pool)
print({k: v for k, v in prep.items() if k not in ("class_counts", "subfolders")})
display(pd.DataFrame(prep["class_counts"]).rename(columns=str.upper))
display(Image(filename=str(prepare.draw_labels(cfg, pool["stem"].iloc[0], FIGS / "labels_mv_vs_st.jpg")), width=900))

### Pseudo-cameras and the official split's leakage

In [ ]:
clu = cluster.run(cfg)
pc, lk, cal = clu["pseudo_cameras"], clu["leakage"], clu["calibration"]
print(f"linking rule: {clu['rule']} | {pc['groups']} pseudo-cameras | {pc['singletons']} singletons | largest {pc['largest_group']} images")
print(f"calibration (equal cameras): {cal['groups']} groups, {cal['singletons_pct_of_pool']}% singletons")
print(f"LEAKAGE: {lk['val_images_sharing_camera_with_train']} of {lk['val_images']} official-val images "
      f"({lk['leakage_val_share']:.1%}) share a pseudo-camera with an official-train image")
if clu["frozen_groups_used"]:
    print("frozen groups used; recomputed grouping identical:", clu["recomputed_groups_identical"])
display(pd.DataFrame(clu["rule_sweep"]).set_index("setting"))

In [ ]:
rule = cluster.Rule.from_config(cfg).name
for name, width in (("nn_gradient_g.png", 640), ("group_sizes.png", 640), (f"{rule}_above.jpg", 900),
                    (f"{rule}_below.jpg", 900), (f"{rule}_weakest_in_group.jpg", 900), ("leak_examples.jpg", 900)):
    if (FIGS / name).exists():  # a grid is skipped when no pair qualifies
        display(Image(filename=str(FIGS / name), width=width))

### Group split and class balance

In [ ]:
if (cfg["paths"]["splits_dir"] / "test.sha256").exists():
    print("split is frozen; test.txt sha256 verified:", split.check_test_checksum(cfg))
    assignment = split.load(cfg)
else:
    assignment = split.make(cfg)
spl = split.summarise(cfg, assignment)
floor = cfg["split"]["min_test_instances"]
for k in ("images", "image_share", "pseudo_cameras", "official_split_composition",
          "classes_missing", f"classes_under_{floor}_test_instances"):
    print(f"{k}: {spl[k]}")
display(pd.read_csv(cfg["paths"]["results_dir"] / "split_class_counts.csv", header=[0, 1], index_col=0))

## 3. Preprocessing and augmentation  *(Phases 1-2)*

## 4. Baseline: YOLOv8s fine-tuned on MV labels  *(Phase 2)*

### Context: COCO-pretrained YOLOv8s as-is
No fine-tuning. Predictions and labels are mapped to shared coarse classes (cars → car; Bus, Mini-bus → bus; Truck, LCV → truck; Two-wheeler → motorcycle; Bicycle → bicycle). Three-wheeler, Tempo-traveller, Van and Others have no COCO class and count as misses. This shows why fine-tuning is needed. Val only.

In [ ]:
from src import evaluate, train

coco = evaluate.coco_context(cfg)
for ls in ("mv", "st"):
    m = coco[ls]
    print(f"{ls.upper()} labels: mAP50 {m['mAP50']:.3f}, mAP50-95 {m['mAP50_95']:.3f} over all coarse classes | "
          f"mappable only: mAP50 {m['mAP50_mappable_only']:.3f}, mAP50-95 {m['mAP50_95_mappable_only']:.3f}")
display(pd.DataFrame(coco["mv"]["per_class"]).T)

### Freeze pilot (val only)
YOLOv8s at 960 px, batch 8, seed 42, MV labels, `pilot_epochs` epochs per arm: backbone frozen (`freeze=10`) vs fully trainable. The choice is made on val mAP@0.50:0.95 and minutes per epoch. Runs are saved to Drive; re-running this cell resumes an interrupted arm and skips a finished one.

In [ ]:
if RUN_TRAINING:
    for arm in ("frozen", "full"):
        train.ensure_run(cfg, "mv", pilot=arm)
pilot = train.pilot_summary(cfg)
display(pd.DataFrame({arm: {
    "val mAP50-95": s["val_last"]["mAP50_95"], "val mAP50": s["val_last"]["mAP50"],
    "minutes / epoch": s["minutes_per_epoch"], "first epoch (min)": s["minutes_first_epoch"],
    "total (min)": s["minutes_total"], f"epochs within {pilot['budget_minutes']} min": s["epochs_within_budget"],
} for arm, s in pilot["arms"].items()}))

### Epoch budget
One full run must fit the budget on a free T4. The epoch count follows from the chosen arm's minutes per epoch (first-epoch overhead and a 10% margin included) and is written to `configs/base.yaml` (`train.epochs`, `train.freeze`), so baseline and ablation use the same value.

In [ ]:
print(f"config now: train.freeze = {cfg['train']['freeze']}, train.epochs = {cfg['train']['epochs']}")
for arm, s in pilot["arms"].items():
    print(f"  if {arm:6s}: freeze = {train.PILOTS[arm]}, epochs = {s['epochs_within_budget']} "
          f"(~{s['minutes_per_epoch']:.2f} min/epoch)")

### Baseline run
Full run on MV labels with the frozen config. The ablation (Section 5) runs the same command with `"st"`; nothing else differs.

In [ ]:
if RUN_TRAINING:
    baseline_dir = train.ensure_run(cfg, "mv")
else:
    baseline_dir = cfg["paths"]["runs_root"] / "mv"
baseline = train.run_summary(baseline_dir)
print({k: baseline[k] for k in ("epochs_run", "minutes_total", "minutes_per_epoch")})
print("val (best epoch):", baseline["val_best"])
display(Image(filename=str(baseline_dir / "results.png"), width=900))

## 5. Ablation: MV vs STAPLE training labels  *(Phase 3)*

## 6. Evaluation  *(Phase 4 threshold on val; Phase 5 sealed test, once)*

## 7. Error analysis  *(Phase 6)*

## 8. Conclusion